In [1]:
"""
AOVE Data Collector — Production-grade data ingestion pipeline
=============================================================================
Downloads and assembles the two CSVs required by aove_predictor.py:

    1. climate_dataset.csv  — weekly climate rows per olive-producing location
    2. macro_dataset.csv    — weekly AOVE prices merged with monthly macro feeds

Data sources
------------
| Dataset             | Provider    | Access          | Cost  |
|---------------------|-------------|-----------------|-------|
| Daily climate       | Open-Meteo  | Public REST API | Free  |
| AOVE origin price   | MAPA scraper| Public PDFs     | Free  |
| Monthly CPI         | INE         | Public REST API | Free  |
| Agricultural diesel | MAPA/MITECO | Annual proxy    | Free  |
| Olive oil stocks    | AICA/MAPA   | Manual CSV      | Free  |

Open-Meteo (climate — NO API KEY REQUIRED):
  Endpoint: https://archive-api.open-meteo.com/v1/archive
  Variables used per location:
    - precipitation_sum          (mm/day  -> aggregated to weekly sum)
    - temperature_2m_max         (°C      -> weekly max)
    - temperature_2m_min         (°C      -> weekly min, used for ETP)
    - et0_fao_evapotranspiration (mm/day  -> aggregated to weekly sum)
  No registration, no API key, no rate limit (fair use).
  Data available from 1940 to present at 9 km resolution (ERA5 reanalysis).

INE CPI (automatic, no key):
  Series IPC251852 — national general index, monthly, base 2021=100.

Agricultural diesel: MITECO annual proxy (replace with real CSV for production).

AICA stocks: seasonal proxy (replace with downloaded CSVs for production).

Usage
-----
    pip install requests pandas numpy

    # Full run
    python data_collector.py \\
        --poolred_csv ./poolred_historico.csv \\
        --start_date 2015-01-01 \\
        --output_dir ./data

    # Climate + CPI only (validate architecture before sourcing prices)
    python data_collector.py --start_date 2015-01-01 --output_dir ./data
"""

import os
import time
import argparse
import logging
import requests
import numpy as np
import pandas as pd
from datetime import date, timedelta
from pathlib import Path
from typing import Optional
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

INE_BASE: str = "https://servicios.ine.es/wstempus/js/ES"
INE_IPC_SERIES: str = "IPC251852"

# ==============================================================================
# OLIVE LOCATIONS
# ==============================================================================
# Each tuple: (display_name, latitude, longitude, productive_surface_ha)
# Coordinates: centroid of the main olive-producing area for each municipality.
# Surface source: ESYRCE 2022 (Encuesta de Superficies y Rendimientos, MAPA).
# Open-Meteo uses these coordinates to select the nearest ERA5 grid cell (9 km).
OLIVE_LOCATIONS: list[tuple[str, float, float, int]] = [
    # Jaen — world's largest olive oil producing province
    ("Jaen capital",    37.779, -3.787, 165_000),
    ("Ubeda",           38.013, -3.370,  98_000),
    ("Baeza",           37.994, -3.469,  72_000),
    ("Linares",         38.095, -3.636,  45_000),
    ("Martos",          37.720, -3.971,  68_000),
    # Cordoba
    ("Cordoba capital", 37.888, -4.779, 140_000),
    ("Lucena",          37.408, -4.486,  88_000),
    ("Cabra",           37.472, -4.443,  52_000),
    # Sevilla
    ("Sevilla capital", 37.389, -5.984,  75_000),
    ("Ecija",           37.542, -5.082,  42_000),
    # Granada
    ("Granada capital", 37.177, -3.598,  55_000),
    ("Baza",            37.494, -2.765,  28_000),
    # Malaga
    ("Antequera",       37.020, -4.559,  35_000),
]


# ==============================================================================
# 1. OPEN-METEO DOWNLOADER
# ==============================================================================
class OpenMeteoDownloader:
    """
    Downloads daily historical weather data from Open-Meteo Historical Weather API.

    Key advantages over the previous implementation:
      - No API key or registration required.
      - Single HTTP call per location per date range (no chunking needed).
      - Returns et0_fao_evapotranspiration directly — no manual ETP calculation.
      - ERA5 reanalysis: complete coverage, no missing station data.
      - 9 km spatial resolution across all of Andalusia.

    API endpoint:
      https://archive-api.open-meteo.com/v1/archive

    Response structure:
      {
        "latitude": 37.78,
        "longitude": -3.8,
        "daily": {
          "time":                        ["2015-01-05", "2015-01-06", ...],
          "precipitation_sum":           [0.0, 2.3, ...],
          "temperature_2m_max":          [14.2, 12.8, ...],
          "temperature_2m_min":          [3.1, 4.5, ...],
          "et0_fao_evapotranspiration":  [1.2, 0.9, ...]
        }
      }

    Water deficit:
      water_deficit_mm = precipitation_sum - et0_fao_evapotranspiration
      (daily, then aggregated to weekly sum)
    """

    ARCHIVE_URL: str = "https://archive-api.open-meteo.com/v1/archive"

    DAILY_VARIABLES: str = (
        "precipitation_sum,"
        "temperature_2m_max,"
        "temperature_2m_min,"
        "et0_fao_evapotranspiration"
    )

    def __init__(self, sleep_between_requests: float = 0.5) -> None:
        self.sleep_s: float = sleep_between_requests
        self.session: requests.Session = requests.Session()
        self.session.headers.update({"Accept": "application/json"})

    def _get(self, params: dict, retries: int = 3) -> Optional[dict]:
        """GET with exponential backoff."""
        for attempt in range(retries):
            try:
                resp = self.session.get(
                    self.ARCHIVE_URL, params=params, timeout=60
                )
                if resp.status_code == 429:
                    wait = 60 * (attempt + 1)
                    logger.warning(f"Open-Meteo rate limit. Waiting {wait}s...")
                    time.sleep(wait)
                    continue
                resp.raise_for_status()
                return resp.json()
            except requests.RequestException as exc:
                logger.warning(f"Attempt {attempt + 1}/{retries} failed: {exc}")
                time.sleep(5 * (attempt + 1))
        return None

    def download_location(
        self,
        name: str,
        lat: float,
        lon: float,
        surface_ha: int,
        start: date,
        end: date,
    ) -> pd.DataFrame:
        """
        Downloads the full date range for one location in a single API call.
        Returns a daily DataFrame with all climate variables.
        """
        params = {
            "latitude":   lat,
            "longitude":  lon,
            "start_date": start.isoformat(),
            "end_date":   end.isoformat(),
            "daily":      self.DAILY_VARIABLES,
            "timezone":   "Europe/Madrid",
            "models": "era5_land"
        }

        logger.info(f"Downloading {name} ({lat}, {lon})...")
        data = self._get(params)
        time.sleep(self.sleep_s)

        if not data or "daily" not in data:
            logger.warning(f"No data returned for {name}")
            return pd.DataFrame()

        daily = data["daily"]
        df = pd.DataFrame({
            "date":        pd.to_datetime(daily["time"]),
            "rainfall_mm": daily["precipitation_sum"],
            "temp_max_c":  daily["temperature_2m_max"],
            "temp_min_c":  daily["temperature_2m_min"],
            "etp_mm":      daily["et0_fao_evapotranspiration"],
        })

        df["water_deficit_mm"] = df["rainfall_mm"] - df["etp_mm"]
        df["surface_ha"]       = surface_ha
        df["station_id"]       = name.lower().replace(" ", "_")

        # Replace None/NaN from API with forward-fill then 0
        df[["rainfall_mm", "temp_max_c", "etp_mm", "water_deficit_mm"]] = (
            df[["rainfall_mm", "temp_max_c", "etp_mm", "water_deficit_mm"]]
            .ffill()
            .fillna(0.0)
        )

        return df[[
            "date", "station_id", "surface_ha",
            "rainfall_mm", "temp_max_c", "water_deficit_mm",
        ]]

    def download_all_locations(self, start: date, end: date) -> pd.DataFrame:
        """
        Downloads all OLIVE_LOCATIONS and aggregates to ISO weekly frequency.

        Aggregation rules (same as the previous implementation):
          - rainfall_mm      : sum  (total weekly precipitation)
          - temp_max_c       : max  (worst-case heat stress of the week)
          - water_deficit_mm : sum  (cumulative weekly P - ETP deficit)

        Week anchor: Monday (ISO week start).
        """
        frames: list[pd.DataFrame] = []

        for name, lat, lon, surface_ha in OLIVE_LOCATIONS:
            df = self.download_location(name, lat, lon, surface_ha, start, end)
            if not df.empty:
                frames.append(df)

        if not frames:
            raise RuntimeError(
                "No climate data retrieved from Open-Meteo. "
                "Check your internet connection."
            )

        df_daily: pd.DataFrame = pd.concat(frames, ignore_index=True)

        # Snap to ISO week (Monday anchor)
        df_daily["date"] = df_daily["date"].dt.to_period("W-SUN").dt.start_time

        df_weekly: pd.DataFrame = (
            df_daily
            .groupby(["date", "station_id", "surface_ha"])[
                ["rainfall_mm", "temp_max_c", "water_deficit_mm"]
            ]
            .agg({
                "rainfall_mm":     "sum",
                "temp_max_c":      "max",
                "water_deficit_mm":"sum",
            })
            .reset_index()
        )

        logger.info(
            f"Climate download complete: {len(df_weekly)} rows "
            f"({df_weekly['date'].nunique()} weeks x "
            f"{df_weekly['station_id'].nunique()} locations)"
        )
        return df_weekly

# ==============================================================================
# 2. INE DOWNLOADER — monthly CPI (automatic, no API key)
# ==============================================================================
class INEDownloader:
    """
    Downloads Spanish national CPI from the public INE JSON API.
    Endpoint: GET /DATOS_SERIE/{code}?date={start}:{end}&tip=A
    """

    @staticmethod
    def download(start: date, end: date) -> pd.DataFrame:
        """Returns DataFrame: reference_date (monthly), ipc_monthly."""
        url = (
            f"{INE_BASE}/DATOS_SERIE/{INE_IPC_SERIES}"
            f"?date={start.strftime('%Y%m%d')}:{end.strftime('%Y%m%d')}&tip=A"
        )
        logger.info("Downloading CPI from INE...")
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        records: list[dict] = resp.json().get("Data", [])
        if not records:
            raise ValueError(f"INE returned no data for series {INE_IPC_SERIES}.")

        rows: list[dict] = [
            {
                "reference_date": pd.Timestamp(r["Fecha"], unit="ms", tz="UTC").tz_convert("Europe/Madrid").replace(day=1, tzinfo=None),
                "ipc_monthly":    float(r["Valor"]),
            }
            for r in records
            if r.get("Fecha") is not None and r.get("Valor") is not None
        ]
        df: pd.DataFrame = (
            pd.DataFrame(rows)
            .sort_values("reference_date")
            .reset_index(drop=True)
        )
        logger.info(f"CPI downloaded: {len(df)} monthly records")
        return df

# ==============================================================================
# 3. POOLRED LOADER — weekly AOVE price (from MAPA PDF scraper output)
# ==============================================================================
class PoolRedLoader:
    """
    Loads the poolred_historico.csv produced by mapa_pdf_scraper.py.

    Expected columns: reference_date, aove_price_eur_kg
    Price is already in EUR/kg (converted during PDF extraction).
    """

    @staticmethod
    def load(csv_path: str) -> pd.DataFrame:
        """Returns DataFrame: reference_date (weekly), aove_price_eur_kg."""
        if not os.path.exists(csv_path):
            raise FileNotFoundError(
                f"AOVE price CSV not found: {csv_path}\n"
                "Run mapa_pdf_scraper.py first to generate it."
            )
        df: pd.DataFrame = pd.read_csv(
            csv_path, sep=None, engine="python", encoding="utf-8-sig"
        )
        df.columns = [c.strip().lower() for c in df.columns]

        # Accept poolred_historico.csv (direct) or any CSV with date + price
        date_col: str = next(
            (c for c in df.columns
             if any(k in c for k in ["reference_date", "fecha", "date", "semana"])),
            df.columns[0],
        )
        price_col: str = next(
            (c for c in df.columns
             if any(k in c for k in ["aove_price", "precio", "price", "eur_kg"])),
            df.columns[1],
        )

        df["reference_date"] = pd.to_datetime(
            df[date_col], dayfirst=True, errors="coerce"
        )
        df["aove_price_eur_kg"] = pd.to_numeric(
            df[price_col].astype(str).str.replace(",", "."),
            errors="coerce",
        )
        df = (
            df[["reference_date", "aove_price_eur_kg"]]
            .dropna()
            .sort_values("reference_date")
            .reset_index(drop=True)
        )
        logger.info(
            f"AOVE prices loaded: {len(df)} weekly records | "
            f"Range: {df['aove_price_eur_kg'].min():.2f}–"
            f"{df['aove_price_eur_kg'].max():.2f} EUR/kg"
        )
        return df

# ==============================================================================
# 4. DIESEL LOADER — monthly agricultural diesel price (semi-manual)
# ==============================================================================
class DieselLoader:
    """
    Loads agricultural diesel (gasoleo B) price for Spain.

    Primary source : MAPA/SGACE weekly report
      https://www.mapa.gob.es/es/ministerio/servicios/
      analisis-y-prospectiva/precios-del-gasoleo-agrario/

    Expected CSV format when --diesel_csv is provided:
      reference_date, diesel_price_eur
      2015-01-01, 0.85

    Fallback: verified annual averages from MITECO reports (2015-2025).
    Low intra-annual variance makes this proxy acceptable for MVP scaler
    fitting; replace with the weekly CSV for production.
    """

    # Annual averages of agricultural diesel ex-VAT (EUR/litre).
    # Source: MITECO monthly reports + MAPA/SGACE weekly bulletins (verified).
    ANNUAL_PROXY: dict[int, float] = {
        # 2010-2014: MITECO historical reports on agricultural diesel prices
        2010: 0.682, 2011: 0.789, 2012: 0.820, 2013: 0.780, 2014: 0.660,
        2015: 0.548, 2016: 0.445, 2017: 0.499, 2018: 0.596,
        2019: 0.549, 2020: 0.394, 2021: 0.598, 2022: 0.950,
        2023: 0.820, 2024: 0.760, 2025: 0.720,
    }

    @staticmethod
    def load(csv_path: Optional[str], start: date, end: date) -> pd.DataFrame:
        """Returns DataFrame: reference_date (monthly), diesel_price_eur."""
        if csv_path and os.path.exists(csv_path):
            df: pd.DataFrame = pd.read_csv(csv_path, parse_dates=["reference_date"])
            df = df[["reference_date", "diesel_price_eur"]].dropna()
            logger.info(f"Diesel loaded from CSV: {len(df)} records")
            return df.sort_values("reference_date").reset_index(drop=True)

        logger.warning("No diesel CSV — using MITECO annual proxy (MVP only).")
        rows: list[dict] = []
        current   = date(start.year, 1, 1)
        end_proxy = date(end.year, 12, 1)
        last_year = max(DieselLoader.ANNUAL_PROXY)
        while current <= end_proxy:
            rows.append({
                "reference_date":  pd.Timestamp(current),
                "diesel_price_eur": DieselLoader.ANNUAL_PROXY.get(
                    current.year, DieselLoader.ANNUAL_PROXY[last_year]
                ),
            })
            current = (
                date(current.year + 1, 1, 1)
                if current.month == 12
                else date(current.year, current.month + 1, 1)
            )
        df = pd.DataFrame(rows)
        logger.info(f"Diesel proxy: {len(df)} monthly records")
        return df

# ==============================================================================
# 5. AICA LOADER — monthly olive oil stocks (semi-manual)
# ==============================================================================
class AICALoader:
    """
    Loads monthly olive oil stock data from AICA / MAPA.

    Primary source (monthly CSVs from October 2016 onwards):
      https://www.mapa.gob.es/es/agricultura/temas/producciones-agricolas/
      aceite-oliva-y-aceituna-mesa/datos_produccion_movimiento_existencias_aica

    One-time setup: download all monthly CSVs to --aica_dir.
    The loader concatenates them and computes stock_delta_pct (MoM %).

    Fallback: annual stock averages from MAPA/COI with a synthetic seasonal
    pattern. Sufficient for architecture validation; replace for production.
    """

    ANNUAL_STOCK_KT: dict[int, int] = {
        # 2010-2014: MAPA annual balance + COI (International Olive Council)
        2010: 520, 2011: 480, 2012: 610, 2013: 520, 2014: 350,
        2015: 620, 2016: 580, 2017: 490, 2018: 510,
        2019: 720, 2020: 840, 2021: 600, 2022: 320,
        2023: 240, 2024: 480, 2025: 650,
    }
    SEASONAL_OFFSET: dict[int, int] = {
        1: 5, 2: 3, 3: 1, 4: -2, 5: -4,  6: -6,
        7: -8, 8: -6, 9: -3, 10: 8, 11: 15, 12: 10,
    }

    @staticmethod
    def load(aica_dir: Optional[str], start: date, end: date) -> pd.DataFrame:
        """Returns DataFrame: reference_date (monthly), stock_delta_pct."""
        if aica_dir and os.path.isdir(aica_dir) and list(Path(aica_dir).glob("*.csv")):
            return AICALoader._from_dir(aica_dir)
        logger.warning("AICA dir absent or empty — using seasonal stock proxy.")
        return AICALoader._proxy(start, end)

    @staticmethod
    def _from_dir(aica_dir: str) -> pd.DataFrame:
        frames: list[pd.DataFrame] = []
        for f in sorted(Path(aica_dir).glob("*.csv")):
            try:
                df = pd.read_csv(f, sep=None, engine="python", encoding="latin-1")
                df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
                frames.append(df)
            except Exception as exc:
                logger.warning(f"Could not parse {f.name}: {exc}")

        df_all: pd.DataFrame = pd.concat(frames, ignore_index=True)
        date_col  = next(
            c for c in df_all.columns
            if any(k in c for k in ["mes", "fecha", "date"])
        )
        stock_col = next(
            c for c in df_all.columns
            if any(k in c for k in ["exist", "stock"])
        )
        df_all["reference_date"] = pd.to_datetime(df_all[date_col], errors="coerce")
        df_all["stock_t"] = pd.to_numeric(
            df_all[stock_col].astype(str).str.replace(",", "").str.replace(".", ""),
            errors="coerce",
        )
        df_clean: pd.DataFrame = (
            df_all[["reference_date", "stock_t"]]
            .dropna()
            .sort_values("reference_date")
            .reset_index(drop=True)
        )
        df_clean["stock_delta_pct"] = df_clean["stock_t"].pct_change() * 100
        logger.info(f"AICA stocks loaded: {len(df_clean)} monthly records")
        return df_clean[["reference_date", "stock_delta_pct"]]

    @staticmethod
    def _proxy(start: date, end: date) -> pd.DataFrame:
        rows: list[dict] = []
        prev: Optional[float] = None
        current   = date(start.year, 1, 1)
        end_proxy = date(end.year, 12, 1)
        last_year = max(AICALoader.ANNUAL_STOCK_KT)
        while current <= end_proxy:
            base  = AICALoader.ANNUAL_STOCK_KT.get(
                current.year, AICALoader.ANNUAL_STOCK_KT[last_year]
            ) * 1_000
            stock = base * (1.0 + AICALoader.SEASONAL_OFFSET.get(current.month, 0) / 100.0)
            delta = ((stock - prev) / prev * 100.0) if prev else 0.0
            rows.append({
                "reference_date": pd.Timestamp(current),
                "stock_delta_pct": delta,
            })
            prev = stock
            current = (
                date(current.year + 1, 1, 1)
                if current.month == 12
                else date(current.year, current.month + 1, 1)
            )
        df = pd.DataFrame(rows)
        logger.info(f"Stock proxy: {len(df)} monthly records")
        return df

# ==============================================================================
# 6. DATASET ASSEMBLER
# ==============================================================================
class DatasetAssembler:
    """
    Combines all feeds into the two production CSVs.

    Frequency alignment:
      The weekly AOVE price series is the chronological backbone of
      macro_dataset.csv. Monthly feeds (CPI, diesel, stocks) are merged
      via merge_asof(direction='backward') — each week receives the last
      published monthly value, replicating real-world information availability
      with no look-ahead bias.
    """

    @staticmethod
    def build_climate_csv(df_climate: pd.DataFrame, output_path: str) -> None:
        """Saves the weekly-per-location climate CSV."""
        df = df_climate.copy()
        for col in ["rainfall_mm", "temp_max_c", "water_deficit_mm"]:
            df[col] = df[col].astype(np.float32)
        df.to_csv(output_path, index=False)
        logger.info(f"climate_dataset.csv -> {output_path}  ({len(df)} rows)")

    @staticmethod
    def build_macro_csv(
        df_prices: pd.DataFrame,   # weekly  : reference_date, aove_price_eur_kg
        df_ipc:    pd.DataFrame,   # monthly : reference_date, ipc_monthly
        df_diesel: pd.DataFrame,   # monthly : reference_date, diesel_price_eur
        df_stock:  pd.DataFrame,   # monthly : reference_date, stock_delta_pct
        output_path: str,
    ) -> None:
        # ── 1. Backbone: weekly prices ────────────────────────────────────────
        backbone: pd.DataFrame = (
            df_prices[["reference_date", "aove_price_eur_kg"]]
            .copy()
            .assign(reference_date=lambda d: pd.to_datetime(d["reference_date"])
                    .astype("datetime64[us]"))   # Canonical resolution for all merges
            .sort_values("reference_date")
            .reset_index(drop=True)
        )

        # ── 2. Merge monthly feeds (no look-ahead) ────────────────────────────
        monthly_feeds: list[tuple[pd.DataFrame, str]] = [
            (df_ipc,    "ipc_monthly"),
            (df_diesel, "diesel_price_eur"),
            (df_stock,  "stock_delta_pct"),
        ]
        for df_feed, col in monthly_feeds:
            feed: pd.DataFrame = (
                df_feed[["reference_date", col]]
                .copy()
                .assign(
                    reference_date=lambda d: pd.to_datetime(d["reference_date"])
                    .astype("datetime64[us]")    # Align resolution with backbone
                )
                .sort_values("reference_date")
                .dropna()
            )
            backbone = pd.merge_asof(
                left=backbone,
                right=feed,
                on="reference_date",
                direction="backward",
            )

        # ── 3. Annual surface variation (ESYRCE, MAPA) ───────────────────────
        surface_annual: dict[int, float] = {
            # 2010-2014: ESYRCE historical reports (MAPA)
            2010: 0.6, 2011: 0.7, 2012: 0.9, 2013: 1.1, 2014: 0.8,
            2015: 0.8, 2016: 1.2, 2017: 0.5, 2018: -0.3,
            2019: 0.9, 2020: 1.1, 2021: 0.4, 2022: -0.2,
            2023: 0.6, 2024: 0.7, 2025: 0.5,
        }
        backbone["surface_delta_pct"] = (
            backbone["reference_date"].dt.year.map(surface_annual)
        )

        # ── 4. Final cleanup ──────────────────────────────────────────────────
        backbone = backbone.sort_values("reference_date").reset_index(drop=True)
        backbone = backbone.ffill().fillna(0.0)

        for col in ["aove_price_eur_kg", "ipc_monthly", "diesel_price_eur",
                    "stock_delta_pct", "surface_delta_pct"]:
            backbone[col] = backbone[col].astype(np.float32)

        backbone.to_csv(output_path, index=False)
        logger.info(
            f"macro_dataset.csv -> {output_path}  ({len(backbone)} weekly rows)\n"
            f"  Columns : {list(backbone.columns)}\n"
            f"  Range   : {backbone['reference_date'].min().date()} -> "
            f"{backbone['reference_date'].max().date()}"
        )

# ==============================================================================
# ENTRY POINT
# ==============================================================================
def _parse_args() -> argparse.Namespace:
    """Jupyter-safe argument parser."""
    parser = argparse.ArgumentParser(
        description="AOVE Data Collector — Open-Meteo edition",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=__doc__,
    )
    parser.add_argument(
        "--poolred_csv", type=str, default=None,
        help="Path to poolred_historico.csv from mapa_pdf_scraper.py.",
    )
    parser.add_argument(
        "--diesel_csv", type=str, default=None,
        help="Monthly diesel price CSV (reference_date, diesel_price_eur). Falls back to MITECO proxy.",
    )
    parser.add_argument(
        "--aica_dir", type=str, default=None,
        help="Directory of monthly AICA stock CSVs. Falls back to seasonal proxy.",
    )
    parser.add_argument(
        "--start_date", type=str, default="2010-01-01",
        help="Download start date YYYY-MM-DD (default: 2010-01-01, ~15 years).",
    )
    parser.add_argument(
        "--end_date", type=str, default=date.today().isoformat(),
        help="Download end date YYYY-MM-DD (default: today).",
    )
    parser.add_argument(
        "--output_dir", type=str, default="./data",
        help="Output directory for the generated CSVs.",
    )

    import sys
    in_jupyter = any("ipykernel" in a or "jupyter" in a for a in sys.argv)
    if in_jupyter:
        logger.info("Jupyter detected — using NOTEBOOK CONFIG defaults.")
        return parser.parse_args([])
    return parser.parse_args()


if __name__ == "__main__":
    # ==========================================================================
    # NOTEBOOK CONFIG — edit when running inside Jupyter
    # ==========================================================================
    NOTEBOOK_POOLRED_CSV = "./data/precio_historico.csv"
    NOTEBOOK_DIESEL_CSV  = None
    NOTEBOOK_AICA_DIR    = None
    NOTEBOOK_START_DATE  = "2010-01-01"
    NOTEBOOK_END_DATE    = "2026-04-27"
    NOTEBOOK_OUTPUT_DIR  = "./data"
    # ==========================================================================

    args = _parse_args()

    import sys
    in_jupyter = any("ipykernel" in a or "jupyter" in a for a in sys.argv)
    if in_jupyter:
        args.poolred_csv = NOTEBOOK_POOLRED_CSV
        args.diesel_csv  = NOTEBOOK_DIESEL_CSV
        args.aica_dir    = NOTEBOOK_AICA_DIR
        args.start_date  = NOTEBOOK_START_DATE
        args.end_date    = NOTEBOOK_END_DATE
        args.output_dir  = NOTEBOOK_OUTPUT_DIR

    start = date.fromisoformat(args.start_date)
    end   = date.fromisoformat(args.end_date)
    weeks = (end - start).days // 7
    Path(args.output_dir).mkdir(parents=True, exist_ok=True)

    logger.info("=" * 60)
    logger.info("AOVE Data Collector (Open-Meteo edition)")
    logger.info(f"  Period : {start} -> {end} ({weeks} weeks)")
    logger.info(f"  Output : {args.output_dir}")
    logger.info("=" * 60)

    # ── 1. Climate (Open-Meteo, no key needed) ───────────────────────────────
    downloader = OpenMeteoDownloader(sleep_between_requests= 6.0)
    df_climate = downloader.download_all_locations(start, end)
    DatasetAssembler.build_climate_csv(
        df_climate,
        os.path.join(args.output_dir, "climate_dataset.csv"),
    )

    # ── 2. CPI (INE, automatic) ──────────────────────────────────────────────
    df_ipc = INEDownloader.download(start, end)

    # ── 3. AOVE price ────────────────────────────────────────────────────────
    if not args.poolred_csv:
        logger.error(
            "No AOVE price CSV provided — macro_dataset.csv will not be generated.\n"
            "Run mapa_pdf_scraper.py first and pass --poolred_csv path/to/poolred_historico.csv"
        )
        raise SystemExit(1)

    df_prices = PoolRedLoader.load(args.poolred_csv)

    # ── 4. Diesel (MAPA/MITECO proxy or CSV) ─────────────────────────────────
    df_diesel = DieselLoader.load(args.diesel_csv, start, end)

    # ── 5. Stocks (AICA or proxy) ────────────────────────────────────────────
    df_stock = AICALoader.load(args.aica_dir, start, end)

    # ── 6. Assemble macro CSV ────────────────────────────────────────────────
    DatasetAssembler.build_macro_csv(
        df_prices, df_ipc, df_diesel, df_stock,
        os.path.join(args.output_dir, "macro_dataset.csv"),
    )

    logger.info("Data Collector finished successfully.")
    logger.info(f"Files: {os.path.abspath(args.output_dir)}/")

INFO: Jupyter detected — using NOTEBOOK CONFIG defaults.
INFO: ============================================================
INFO: AOVE Data Collector (Open-Meteo edition)
INFO:   Period : 2010-01-01 -> 2026-04-27 (851 weeks)
INFO:   Output : ./data
INFO: ============================================================
INFO: Downloading Jaen capital (37.779, -3.787)...
C:\Users\juanm\AppData\Local\Temp\ipykernel_9536\2427918725.py:210: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .ffill()
C:\Users\juanm\AppData\Local\Temp\ipykernel_9536\2427918725.py:211: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_opti